# De una pendiente a un resultado comprobado

Lee el flujo completo de un programa: entradas, función, integración, forma del resultado y verificación. Después compara un Jacobiano analítico con diferencias centrales.

Lecciones: 00 · Retomar las ecuaciones diferenciales: del balance a la solución, 02 · Programar y resolver modelos con Python: algoritmos, precisión y rigidez, 05 · Equilibrios y estabilidad: linealización, autovalores y retroalimentación.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A27](https://math.mit.edu/~gs/linearalgebra/ila6/indexila6.html), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [R20](https://docs.python.org/3/tutorial/index.html), [R21](https://numpy.org/doc/stable/user/basics.indexing.html).

## Lectura del código

La función pendientes recibe un vector de estado y devuelve otro con la misma longitud. El solver construye la trayectoria llamándola en estados internos de prueba. La salida se compara con una fórmula derivada a mano, mediante error absoluto y tolerancias explícitas. En la última celda, diferencias centrales generan una columna del Jacobiano por cada entrada modificada. El ejemplo triangular permite conocer los autovalores antes de pedirlos a NumPy y sirve como control independiente.

## Preparación
El primer bloque localiza los módulos del paquete. `__file__` existe al ejecutar un script; en Jupyter se utiliza la carpeta actual. `sys.path.insert` permite importar el módulo de modelos. Ejecuta desde `python/cuadernos/` o desde `python/`. Lee los comentarios antes de cada llamada.

In [ ]:
# Localizar los módulos en python/, tanto como script como en Jupyter.
from pathlib import Path
import sys
base = Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd()
if not (base / 'circuitos_casos.py').exists():
    base = base.parent
sys.path.insert(0, str(base))
# NumPy organiza vectores; SciPy integra; Matplotlib dibuja resultados.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import circuitos_casos as cc


## Definir el balance y nombrar cada término
Antes de ejecutar, identifica qué entra, qué sale y las unidades. El capítulo correspondiente deriva las fórmulas que se están comprobando.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A27](https://math.mit.edu/~gs/linearalgebra/ila6/indexila6.html), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [R20](https://docs.python.org/3/tutorial/index.html), [R21](https://numpy.org/doc/stable/user/basics.indexing.html).

In [ ]:
def pendientes(t, estado, alfa, delta):
    x = estado[0]             # una concentración en nM
    produccion = alfa        # nM/h; independiente del estado
    perdida = delta * x      # nM/h; aumenta con x
    return np.array([produccion - perdida])

tiempos = np.linspace(0, 20, 201)  # 201 salidas, separadas 0,1 h
inicial = [5.0]                   # el vector tiene una componente
parametros = (12.0, 0.3)          # orden: alfa, delta
sol = solve_ivp(pendientes, (0, 20), inicial, args=parametros,
                t_eval=tiempos, rtol=1e-9, atol=1e-11)
if not sol.success:
    raise RuntimeError(sol.message)
print('Forma especie/tiempo:', sol.y.shape)


## Comparar con una solución obtenida por integración algebraica
Antes de ejecutar, identifica qué entra, qué sale y las unidades. El capítulo correspondiente deriva las fórmulas que se están comprobando.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A27](https://math.mit.edu/~gs/linearalgebra/ila6/indexila6.html), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [R20](https://docs.python.org/3/tutorial/index.html), [R21](https://numpy.org/doc/stable/user/basics.indexing.html).

In [ ]:
exacta = cc.primer_orden(tiempos, alfa=12, delta=.3, x0=5)
error = np.abs(sol.y[0, :] - exacta)  # una discrepancia por tiempo
print('Máximo error absoluto (nM):', error.max())
np.testing.assert_allclose(sol.y[0], exacta, atol=1e-7, rtol=1e-7)
plt.plot(tiempos, sol.y[0], label='Integración')
plt.plot(tiempos, exacta, '--', label='Solución analítica')
plt.xlabel('Tiempo (h)'); plt.ylabel('Concentración (nM)')
plt.legend(); plt.show()


## Construir columnas de derivadas moviendo una variable cada vez
Antes de ejecutar, identifica qué entra, qué sale y las unidades. El capítulo correspondiente deriva las fórmulas que se están comprobando.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A27](https://math.mit.edu/~gs/linearalgebra/ila6/indexila6.html), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [R20](https://docs.python.org/3/tutorial/index.html), [R21](https://numpy.org/doc/stable/user/basics.indexing.html).

In [ ]:
def dos_pendientes(estado):
    m, p = estado
    return np.array([4 - .5*m, 2*m - .2*p])

# Cada columna responde a cambiar una entrada, con las otras fijas.
analitico = np.array([[-.5, 0], [2, -.2]])
numerico = cc.diferencia_central(dos_pendientes, [3, 8], paso=1e-5)
print('Jacobiano calculado:', numerico)
np.testing.assert_allclose(numerico, analitico, atol=1e-8)
# Los autovalores de una matriz triangular son sus diagonales.
print('Tasas locales:', np.linalg.eigvals(analitico))


## Comprueba con tus palabras
Explica por qué 201 tiempos no significan 201 pasos internos. Intercambia alfa y delta, observa que Python puede ejecutarlo y localiza el error de unidades. Deriva las cuatro entradas del Jacobiano antes de ejecutar la última celda.

Los `assert_allclose` comparan consecuencias analíticas o implementaciones equivalentes; no validan experimentalmente las hipótesis.